In [ ]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, os, random
import numpy as np
import pandas as pd
import warnings
import fastf1

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

cache_path = os.path.join(os.getcwd(), 'data', 'fastf1_cache')
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'fastf1  : {fastf1.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print(f'Cache   : {cache_path}')

import requests
warnings.filterwarnings('ignore')

def fetch_results(seasons):
    rows = []
    for season in seasons:
        offset = 0
        while True:
            url = f'https://api.jolpi.ca/ergast/f1/{season}/results.json?limit=100&offset={offset}'
            resp = requests.get(url)
            data = resp.json()
            races = data['MRData']['RaceTable']['Races']
            if not races:
                break
            for race in races:
                for r in race['Results']:
                    pos_num = int(r['position']) if r['position'].isdigit() else None
                    status = r['status']
                    rows.append({
                        'season': int(race['season']),
                        'round': int(race['round']),
                        'race': race['raceName'],
                        'date': race['date'],
                        'circuit': race['Circuit']['circuitId'],
                        'driverId': r['Driver']['driverId'],
                        'driver': f"{r['Driver']['givenName']} {r['Driver']['familyName']}",
                        'constructor': r['Constructor']['name'],
                        'position': pos_num,
                        'positionText': r['positionText'],
                        'grid': int(r['grid']),
                        'laps': int(r['laps']),
                        'status': status,
                        'points': int(float(r['points'])),
                        'top10': pos_num is not None and pos_num <= 10,
                        'finished': status == 'Finished' or 'Lap' in status,
                    })
            total = int(data['MRData']['total'])
            offset += 100
            if offset >= total:
                break
    return pd.DataFrame(rows)

cache_file = os.path.join('data', 'processed', 'results_2022_2024.csv')
os.makedirs(os.path.dirname(cache_file), exist_ok=True)

if os.path.exists(cache_file):
    df = pd.read_csv(cache_file, parse_dates=['date'])
    df['top10'] = df['top10'].astype(bool)
    df['finished'] = df['finished'].astype(bool)
else:
    df = fetch_results([2022, 2023, 2024])
    df['date'] = pd.to_datetime(df['date'])
    df.to_csv(cache_file, index=False)

df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')
df['top10'] = df['top10'].astype(bool)

print(f'Total rows: {len(df)}')
print(f'Seasons: {sorted(df["season"].unique())}')
df.head()

# Baseline: Domain Heuristic + Stretch

This notebook implements a rule-based baseline (no ML) and optional stretch metrics/models. Rule: if `grid <= 10`, predict top-10 finish. We evaluate on the validation set (season 2023).

In [ ]:
train_df = df[df['season'] == 2022].copy()
val_df = df[df['season'] == 2023].copy()
test_df = df[df['season'] == 2024].copy()

val_df['grid'] = pd.to_numeric(val_df['grid'], errors='coerce')
val_df = val_df.dropna(subset=['grid'])

val_df['pred_top10'] = val_df['grid'] <= 10

accuracy = (val_df['pred_top10'] == val_df['top10']).mean()
always_top10_acc = val_df['top10'].mean()

print(f'Validation rows: {len(val_df):,}')
print(f'Heuristic (grid<=10) accuracy on validation: {accuracy:.4f}')
print(f'Always-predict-top10 accuracy on validation: {always_top10_acc:.4f}')

confusion = pd.crosstab(val_df['top10'], val_df['pred_top10'], rownames=['actual_top10'], colnames=['pred_top10'])
print('\nConfusion table (rows=actual, cols=pred):')
print(confusion)

## Reflection on accuracy

The heuristic reached ~74% accuracy on the validation set. Compared to always predicting top-10 (~50% accuracy), it adds ~24 percentage points of signal, so it clearly beats random guessing.

However, accuracy alone can be misleading. If the dataset were imbalanced (say 80% top-10), a naive model that always predicts top-10 would score 80% while being useless. Even with balanced classes, accuracy hides whether errors are concentrated in false positives or false negatives. That is why we compute precision, recall, and F1 below.

**Any model we build in Lab 2 must beat this baseline accuracy. If it doesn't, the model adds no value.**

## Stretch: Additional metrics and sklearn baseline

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

y_true = val_df['top10'].astype(int).values
y_pred = val_df['pred_top10'].astype(int).values

print('=== Heuristic baseline (grid <= 10) ===')
print(classification_report(y_true, y_pred, target_names=['Non-Top10', 'Top10']))

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-score:  {f1:.4f}')

Precision tells us what fraction of our top-10 predictions were actually correct, and recall tells us what fraction of actual top-10 finishes we caught. F1 balances both. These give a fuller picture than accuracy alone, since different types of mistakes have different costs depending on the decision context.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

train_df_clean = train_df.dropna(subset=['grid']).copy()
val_df_clean = val_df.dropna(subset=['grid']).copy()

X_train = train_df_clean[['grid']].values
y_train = train_df_clean['top10'].astype(int).values
X_val = val_df_clean[['grid']].values
y_val = val_df_clean['top10'].astype(int).values

dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_val)

lr = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_val)

print('=== DummyClassifier (most_frequent) ===')
print(classification_report(y_val, dummy_preds, target_names=['Non-Top10', 'Top10']))

print('=== Logistic Regression (grid only) ===')
print(classification_report(y_val, lr_preds, target_names=['Non-Top10', 'Top10']))

print(f'Heuristic accuracy: {accuracy:.4f}')
print(f'Dummy accuracy:     {(dummy_preds == y_val).mean():.4f}')
print(f'LogReg accuracy:    {(lr_preds == y_val).mean():.4f}')

## Metric choice justification

For this problem, I think **F1-score** matters most. Since the classes are roughly balanced, accuracy is a decent starting point, but F1 penalizes models that sacrifice precision for recall or vice versa. In a real scenario, we might care more about recall (not missing actual top-10 finishers) or precision (not giving false confidence), depending on the decision context. For now, F1 gives us a balanced view.

The logistic regression with just grid position should perform similarly to the heuristic since both rely on the same feature, but it learns the optimal threshold from data rather than using a fixed cutoff of 10.

**Lower bound: Any Lab 2 model must beat the heuristic baseline metrics above. If it doesn't, the model adds no value over a simple grid-position rule.**